In [ ]:
import sys
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer

#load data
data=load_breast_cancer(as_frame=True)
X=data.data #features
y=data.target # label





In [ ]:
# display data
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",20)
df=data.frame
df
print("X shape:", X.shape)
print("y shape:", y.shape)


In [38]:
# data checks
list(X.columns) 
X.dtypes.value_counts() #no text, no encoding 
X.isnull().sum() # null per col
X.isnull().sum().sum() #total of nulls through all cols.
data.target_names
y.value_counts() # class distribution : 0->212 1->357 , tarin/test should keep the same 63/37 % mix
X.describe() # scales 
X[["mean radius", "mean area", "mean smoothness"]].describe()

,mean radius,mean area,mean smoothness
count,569.000000,569.000000,569.000000
mean,14.127292,654.889104,0.096360
std,3.524049,351.914129,0.014064
min,6.981000,143.500000,0.052630
25%,11.700000,420.300000,0.086370
50%,13.370000,551.100000,0.095870
75%,15.780000,782.700000,0.105300
max,28.110000,2501.000000,0.163400


In [ ]:
#split train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,stratify=y)

print("Train: ", X_train.shape)

print("Test: " , X_test.shape)

print("Train classes:", y_train.value_counts(normalize=True)) # verify 0's and 1's in train set 63/37 % mix
print("Test classes:", y_test.value_counts(normalize=True))  # verify 0's and 1's in test set 63/37 % mix



In [ ]:
#Train
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipeline=Pipeline([ ("scaler",StandardScaler()),
("model",LogisticRegression(max_iter=1000,random_state=42))
])
pipeline.fit(X_train,y_train)
pipeline

In [57]:
#Predict
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
y_predict=pipeline.predict(X_test)
accuracy_score(y_test,y_predict)
confusion_matrix(y_test, y_predict)


array([[39,  3],
       [ 1, 71]])

In [59]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predict, target_names=data.target_names))

              precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [60]:
sample = X_test.iloc[[0]]

pred = pipeline.predict(sample)[0]
real = y_test.iloc[0]

print("Predicted:", data.target_names[pred])
print("Real:     ", data.target_names[real])

Predicted: benign
Real:      benign


## Baseline results

- Model: StandardScaler + LogisticRegression (max_iter=1000) in a Pipeline
- Split: 80/20, stratified (455 train / 114 test)
- Accuracy: 0.96
- Malignant: precision 0.97, recall 0.93 (3 of 42 cancers missed)
- Benign: precision 0.96, recall 0.99
- Sample prediction: first test patient predicted benign, really benign

**Do results make sense?**
 Yes: shapes are correct (30 features in, one label out),
both classes are predicted well, and no leakage (scaler fitted on training data only).
Weak spot: 3 missed malignant cases.